# 10 — Statistics pipeline

**Workload:** Histograms, percentiles/quantiles, NaN-aware statistics, and polynomial fitting.

This notebook is executed against the RNP engine. Every output below is
stored in the notebook and visible when rendered on GitHub.

In [1]:
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "shim" / "rnp_numpy").is_dir()
)
for path in (PROJECT_ROOT / "harness" / "_redirect", PROJECT_ROOT / "shim"):
    sys.path.insert(0, str(path))

# IPython may preload the oracle NumPy, so clear that namespace before
# executing the exact redirect hook used by examples/run_all.py.
for module_name in list(sys.modules):
    if module_name == "numpy" or module_name.startswith("numpy."):
        del sys.modules[module_name]
redirect_path = PROJECT_ROOT / "harness" / "_redirect" / "sitecustomize.py"
redirect_spec = importlib.util.spec_from_file_location("_rnp_notebook_redirect", redirect_path)
redirect = importlib.util.module_from_spec(redirect_spec)
redirect_spec.loader.exec_module(redirect)
import numpy as np

probe = np.array(0)
print("numpy version:", np.__version__)
print(f"RNP engine active: {np.__name__} ({type(probe).__module__}.{type(probe).__name__})")
assert np.__name__ == "rnp_numpy"

numpy version: 2.5.2
RNP engine active: rnp_numpy (_rnp.ndarray)


## Summarize measurements with missing values

Build a histogram and calculate percentile, quantile, and NaN-aware statistics.

In [2]:
measurements = np.array([
    1.0, 2.0, np.nan, 4.0, 5.0, 7.0, 8.0, np.nan, 10.0, 12.0,
])
valid = measurements[~np.isnan(measurements)]
histogram, bin_edges = np.histogram(valid, bins=np.array([0.0, 3.0, 6.0, 9.0, 12.1]))
percentiles = np.nanpercentile(measurements, [10.0, 50.0, 90.0])
quantiles = np.nanquantile(measurements, [0.25, 0.75])
nan_summary = np.array([
    np.nanmean(measurements), np.nanstd(measurements),
    np.nanmin(measurements), np.nanmax(measurements),
])
print("histogram:", histogram)
print("bin edges:", bin_edges)
print("10/50/90 percentiles:", percentiles)
print("25/75% quantiles:", quantiles)
print("nan-aware mean/std/min/max:", np.round(nan_summary, 6))

histogram: [2 2 2 2]
bin edges: [ 0.   3.   6.   9.  12.1]
10/50/90 percentiles: [ 1.7  6.  10.6]
25/75% quantiles: [3.5 8.5]
nan-aware mean/std/min/max: [ 6.125     3.585997  1.       12.      ]


## Fit a quadratic trend

RNP's legacy `np.polyfit` wrapper still reaches an unimplemented
private binding. As documented in [KNOWN_GAPS.md](../KNOWN_GAPS.md),
this uses the supported `np.polynomial.polynomial.polyfit` API.

In [3]:
x = np.arange(7, dtype=np.float64)
y = 1.5 * x * x - 0.5 * x + 2.0
polyfit = np.polynomial.polynomial.polyfit(x, y, deg=2)[::-1]
print("quadratic coefficients:", np.round(polyfit, 6))

quadratic coefficients: [ 1.5 -0.5  2. ]


## Verify the result

In [4]:
assert np.array_equal(histogram, [2, 2, 2, 2])
assert np.allclose(percentiles, [1.7, 6.0, 10.6], rtol=0.0, atol=1e-12)
assert np.allclose(polyfit, [1.5, -0.5, 2.0], rtol=0.0, atol=1e-12)
print("PASS — all statistics-pipeline assertions passed.")

PASS — all statistics-pipeline assertions passed.
